### Developing human heart - regions in space and time (mp_heart_dvlp02)

**Paper**: Lázár E, Mauron R, Andrusivová Ž, Foyer J, He M, Larsson L, Shakari N, Salas SM, Avenel C, Sariyar S, Hansen JN, Vicari M, Czarnewski P, Braun E, Li X, Bergmann O, Sylvén C, Lundberg E, Linnarsson S, Nilsson M, Sundström E, Adameyko I, Lundeberg J. *Spatiotemporal gene expression and cellular dynamics of the developing human heart.* Nat Genet. 2025 Nov;57(11):2756-2771. doi: [10.1038/s41588-025-02352-6](https://doi.org/10.1038/s41588-025-02352-6). PMID: 41162788; PMCID: PMC12597827.

Code: [github.com/rmauron/HDCA_heart_dev](https://github.com/rmauron/HDCA_heart_dev) (Zenodo 15912657) - Data: Mendeley [part 1](https://doi.org/10.17632/fhtb99mdzd.1) (Space Ranger, Cell Ranger, metadata) and [part 2](https://doi.org/10.17632/w65jtfsvpr.1) (Seurat RDS) - [Interactive viewer](https://hdcaheart.serve.scilifelab.se/web/index.html)

It continues **mp_heart_dvlp01_prepare.ipynb**: the regions computed there (**heart01_region_labels.tsv**: Leiden clusters of the cell-type abundances of the 38 sections) are read, not recomputed, and studied in depth: how they match the paper's 23 spatial clusters, which cell types define them, and how the tissue changes between PCW 6 and 12.

#### Libs: **src/libs**

- **single_cell_lib**
  - Class **SingleCell**: generic Scanpy, Squidpy, cell2location and LIANA methods
  - **DEFAULT_PARAMS**
- **single_cell_heart_dvlp_lib** (new)
  - Class **SingleCellHeart(SingleCell)**: developing-heart methods (sections and ages, the paper's stereoscope proportions and spatial clusters, gene x age panels, regions vs the paper, temporal trends)
  - **HEART_PARAMS**, **HEART_LINEAGE** (31 cell types in 9 lineages), **HEART_GENES** (20 heart-development genes), **HEART_FILES** (one file registry per notebook: 'heart01' ... 'heart04')

| section | content |
|---|---|
| 1 | spots, abundances and regions of mp_heart_dvlp01 |
| 2 | region table: size, sections, ages, cell types, best paper cluster |
| 3 | regions vs the paper's 23 spatial clusters (17 sections) |
| 4 | cell types in each region (enrichment heatmap) |
| 5 | regions by age: share per section, mean per age; maps |
| 6 | cell-type trends with age (31 types, sections as replicates) |
| 7 | fine-grained states (paper's stereoscope, 74 states): conduction system, innervation, chromaffin cells |
| 8 | the 20 heart-development genes per region (dot plot) |
| 9 | NMF co-location (cell2location map only) |
| 10 | interactive region maps |

Figures and tables have the prefix **heart02_**.

## Loading packages

In [ ]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import gc

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42          # editable text in PDFs
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'dpi': 90}   # inline display only; files are saved at 300 dpi
import plotly.io as pio
pio.renderers.default = "notebook_connected"

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_heart_dvlp_lib import SingleCellHeart, HEART_PARAMS, HEART_LINEAGE, HEART_GENES, HEART_FILES
from libs.single_cell_lib import cmap_byr, cmap_lbwr
from libs.matplotlib_figure_style import apply_figure_style
from libs.Basic import create_dir

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 250)
pd.set_option('display.max_rows', 200)
print({p: version(p) for p in ['scanpy', 'squidpy', 'anndata', 'cell2location', 'plotly']})

## Parameters

**HEART_PARAMS** + **HEART_ANALYSIS_PARAMS['heart02']** (NMF: 12 factors), **HEART_FILES['inputs']** + **['heart02']**. Override with `SingleCellHeart(ROOT0, analysis='heart02', params=dict(...))`.

In [ ]:
force=False
verbose=True

apply_figure_style()
sch = SingleCellHeart(ROOT0, analysis='heart02')
root_data, root_figure, root_result = sch.root_data, sch.root_figure, sch.root_result
run_name = sch.file_path('c2l_map_dir')
print(sch)
print("cell2location map:", sch.file_path('c2l_map').exists(), sch.file_path('c2l_map'))
df_params = sch.calc_df_params()
df_params[df_params.value.astype(str) != df_params.default.astype(str)]

### 1. Spots, abundances and the regions of mp_heart_dvlp01

The abundance source must be the one the regions were made with (column **abundance_source** of **heart01_region_labels.tsv**): with the cell2location map present, the cell2location map is read (**read_h5ad_slots**, no layers); otherwise the Visium h5ad with the paper's stereoscope proportions.

In [ ]:
df_regions = sch.read_tsv('region_labels', verbose=verbose)
source = df_regions['abundance_source'].iloc[0]
print("regions of mp_heart_dvlp01 made with:", source)

adata_vis = sch.read_h5ad_slots('c2l_map', skip=('layers',), verbose=verbose) if source == 'c2l' else \
            sch.read_h5ad('visium', set_self=False, verbose=verbose)
if source == 'c2l':       # the map has no stereoscope / paper columns: add them from the Visium h5ad
    adata_src = sch.read_h5ad('visium', set_self=False)
    for c in ['paper_cluster', 'paper_cluster_name', 'in_paper_qc']:
        adata_vis.obs[c] = adata_src.obs.loc[adata_vis.obs_names, c].values
    for k in ['stereoscope_hl', 'stereoscope_dl']:
        adata_vis.obsm[k] = pd.DataFrame(adata_src.obsm[k]).loc[adata_vis.obs_names]
    del adata_src
sch.calc_qc_metrics(adata_vis)
df_abund = sch.get_abundance(adata_vis, source=source)
names_auto = sch.add_region_labels_from_table(df_regions, adata_vis)
df_region_info = sch.read_region_info(source=source)
names = dict(zip(df_region_info.region_id, df_region_info.region_name)) if df_region_info is not None else names_auto
regions = list(adata_vis.obs['region_label'].cat.categories)
sections_rep = sch.representative_sections()
df_sections = sch.section_table(adata_vis)
print(adata_vis.shape, '|', len(regions), 'regions |', df_abund.attrs['source'], df_abund.shape)

In [ ]:
df_region_info if df_region_info is not None else pd.Series(names_auto, name='region_name')

### 2. Region table

Per region: spots, sections, ages present, median UMIs, the cell types with the largest share and the largest enrichment (region mean / all spots), and the best matching paper cluster (section 3).

In [ ]:
force=False
verbose=True
df_region_summary = sch.calc_df_region_summary(adata_vis, df_abund, names, fname=sch.fname('tsv_region_summary'), force=force, verbose=verbose)
df_region_summary

### 3. Regions vs the paper's 23 spatial clusters

The paper clustered 25,309 spots of the **17 sections with at least three chambers** on gene expression (23 clusters, e.g. compact / trabecular myocardium of each ventricle, atria, AV plane, valves, epicardium, great vessels; **extended_table_12**). Here each region is compared with them on the same spots: share of the region's spots in every paper cluster, **paper_best** (share >= **region_min_overlap** = 0.30, otherwise **mixed: A / B**), adjusted Rand index (ARI) and normalised mutual information (NMI).

**What to expect.** The two partitions answer different questions: the regions group spots by **cell-type composition** (deconvolution of 31 types), the paper's clusters by **expression** (thousands of genes, including chamber-specific genes inside one cell type). Compact LV and RV myocardium have the same composition (Mat_vCM) and differ by genes such as *HAND1* / *IRX* genes, so they share a region; structures defined by a distinct cell type (valve mesenchyme, tunica media, epicardium, ganglia) match closely.

In [ ]:
force=False
verbose=True
df_vs = sch.calc_df_region_vs_paper(adata_vis, verbose=verbose)
print(f"spots compared: {df_vs.attrs['n_spots']:,}; ARI {df_vs.attrs['ARI']:.3f}; NMI {df_vs.attrs['NMI']:.3f}")
df_vs[['n_shared', 'paper_best', 'paper_share']].assign(region_name=lambda d: d.index.map(names))

In [ ]:
force=False
verbose=True
fig = sch.plot_region_vs_paper(df_vs, names, figsize=(18, 11), fname=sch.fname('fig_region_vs_paper'), force=force, verbose=verbose)
plt.show()

#### The paper's clusters on the same sections

One section per age among the 17 clustered sections (w7.5 and w11 have none).

In [ ]:
sec17 = df_sections[df_sections.in_paper_clustering.astype(bool)]
sec17_rep = sec17.sort_values(['age_pcw', 'n_chambers', 'n_spots'], ascending=[True, False, False]).groupby('age_label', sort=False).head(1).section_id.tolist()
full = sch.paper_cluster_full_names()
fig = sch.plot_regions_by_age(adata_vis, key='paper_cluster_name', sections=sec17_rep, names=full, ncols=len(sec17_rep), panel_size=3.4,
                              legend_ncol=4, fname=None)
plt.show()
fig = sch.plot_regions_by_age(adata_vis, key='region_label', sections=sec17_rep, names=names, ncols=len(sec17_rep), panel_size=3.4,
                              legend_ncol=3, fname=None)
plt.show()

### 4. Cell types in each region

log2 enrichment of every cell type in each region (region mean / mean over all spots; clipped at 1/16). Cell-type labels are coloured by lineage.

In [ ]:
force=False
verbose=True
df_enrich = sch.plot_region_celltype_heatmap(adata_vis, df_abund, names=names, figsize=(17, 10), fname=sch.fname('fig_region_celltype'),
                                             force=force, verbose=verbose)
plt.show()

### 5. Regions by age

The share of each region per section, then the mean over the sections of each age (each section weighs the same; n = sections). The section's plane and size change the shares (e.g. w11-w12 sections show only part of the heart), so read the trends together with the per-section values (**share_sd**).

In [ ]:
force=False
verbose=True
df_region_age = sch.calc_df_region_age(adata_vis, fname=sch.fname('tsv_region_age'), force=force, verbose=verbose)
fig = sch.plot_region_age(df_region_age, names=names, figsize=(16, 7), fname=sch.fname('fig_region_age'), force=force, verbose=verbose)
plt.show()

In [ ]:
fig = sch.plot_group_age_heatmap(df_region_age, group_col='region', figsize=(9, 8), title='Region share by age (row-scaled)')
plt.show()
df_region_age.pivot_table(index='region', columns='age_label', values='share_mean', observed=True).round(3)

In [ ]:
force=False
verbose=True
fig = sch.plot_regions_by_age(adata_vis, key='region_label', sections=sections_rep, names=names, ncols=5, panel_size=3.6,
                              legend_ncol=3, fname=sch.fname('fig_regions_age'), force=force, verbose=verbose)
plt.show()

In [ ]:
# interactive version (HTML in heart_dvlp/figures): click a region to hide it, double-click to isolate it in every section
fig = sch.plotly_regions_by_age(adata_vis, key='region_label', sections=sections_rep, names=names, ncols=5,
                                fname='fig_regions_age_html', force=force, verbose=verbose)
fig.show()

### 6. Cell-type trends with age

Mean share of each cell type over the spots of a section (abundance / spot total), points = sections, line = mean per age. **Spearman rho** between the share and the PCW over the 38 sections gives the direction; with 16 hearts the p-values describe the sections, not independent hearts.

In [ ]:
force=False
verbose=True
df_trend = sch.calc_df_celltype_age_trend(adata_vis, df_abund, fname=sch.fname('tsv_celltype_trend'), force=force, verbose=verbose)
fig = sch.plot_celltype_age_trend(df_trend, ncols=3, fname=sch.fname('fig_celltype_trend'), force=force, verbose=verbose)
plt.show()

In [ ]:
from scipy.stats import spearmanr
rows = []
for ct, d in df_trend.groupby('cell_type', sort=False):
    rho, p = spearmanr(d.age_pcw, d.share)
    rows.append(dict(cell_type=ct, lineage=d.lineage.iloc[0], mean_share=d.share.mean(), rho_vs_pcw=rho, p=p))
df_rho = pd.DataFrame(rows).sort_values('rho_vs_pcw')
df_rho.round(4)

### 7. Fine-grained states: conduction system, innervation and chromaffin cells

The paper's deep-level stereoscope proportions (**obsm['stereoscope_dl']**, 74 states) resolve states that the 31 high-level types merge: **SAN_CM** (sinoatrial node), **AVN_CM** (atrioventricular node), **AVB-BB_CM** (AV bundle - bundle branches), **PF_CM** / **TsPF_CM** (Purkinje fibres, transitional), **Cond_aCM** (atrial conduction), autonomic neurons (**Aut_Neu_1/2**), Schwann cell precursors (**SCP_1-5**) and **Chrom_C** (chromaffin cells, first described in the fetal heart by the paper). Maps on one section per age; one colour scale per state.

In [ ]:
df_dl = pd.DataFrame(adata_vis.obsm['stereoscope_dl'], index=adata_vis.obs_names)
states = ['SAN_CM', 'AVN_CM', 'AVB-BB_CM', 'PF_CM', 'Cond_aCM', 'Aut_Neu_1', 'Chrom_C', 'SCP_1']
force=False
verbose=True
fig = sch.plot_spatial_celltypes_by_age(states, adata_vis, df_dl, sections=sections_rep, panel_size=2.6, cbar_label='stereoscope proportion',
                                        fontsize_scale=1.2, fname=sch.fname('fig_conduction'), force=force, verbose=verbose)
plt.show()

In [ ]:
# where are they? mean proportion per region (x 1000) and the region with the maximum
df_state_region = (df_dl[states].groupby(adata_vis.obs['region_label'].values).mean() * 1000).round(2)
df_state_region.loc['top region'] = [df_state_region[c].astype(float).idxmax() for c in states]
df_state_region

### 8. The 20 heart-development genes per region

Fraction of spots with counts > 0 (dot size) and mean log1p(CP10k) scaled to the maximum over the regions (colour). Spots are mixtures, so a gene is shared by neighbouring regions; read the colour (relative level).

In [ ]:
force=False
verbose=True
genes_plot = list(HEART_GENES)
order = [r for r in regions if r not in sch.params['subregion_skip']]
df_frac, df_mean, df_rel = sch.calc_df_dotplot(genes_plot, 'region_label', adata_vis, order=order)
gene_groups = [('TF', ['NKX2-5', 'GATA4', 'TBX5', 'TBX20']), ('heart fields', ['HAND1', 'HAND2', 'ISL1']),
               ('atrial', ['MYH6', 'MYL7', 'NPPA']), ('ventricular', ['MYH7', 'MYL2', 'IRX4']),
               ('L-R / RA', ['PITX2', 'BMP10']), ('conduction', ['TBX3', 'SHOX2', 'HCN4']), ('epicardium', ['WT1', 'TCF21'])]
fig = sch.plot_dotplot(df_frac, df_rel, gene_groups=gene_groups, row_labels={r: f"{r} {names.get(r, '')}" for r in order},
                       cmap='RdPu', figsize=(17, 11), title='Heart-development genes per region',
                       fname=sch.fname('fig_region_genes'), force=force, verbose=verbose)
plt.show()

### 9. NMF co-location

cell2location **run_colocation** on the q05 abundances (12 factors, **coloc_n_facts**); needs the cell2location map.

In [ ]:
force=False
verbose=True
if source == 'c2l':
    adata_vis, res_dict, coloc_edited = sch.run_colocation_cached(adata_vis, run_name)
    for n in sch.params['coloc_n_facts']:
        fig, df_load = sch.plot_factor_dotplot(adata_vis, n, lineage=HEART_LINEAGE, fname=sch.fname('fig_factor_dotplot', n=n),
                                               fname_tsv=sch.fname('tsv_factor_load', n=n), force=force, verbose=verbose)
        plt.show()
else:
    print("NMF co-location needs the cell2location map: skipped (stereoscope abundances)")

### 10. Interactive region maps

Hover: region name, top cell types (share) and best paper cluster. One HTML per representative section in **heart_dvlp/figures** (**heart02_region_map_<section>.html**); the w8 section is shown here.

In [ ]:
force=False
verbose=False
hover = {r: f"{row.top_share}<br>paper: {row.paper_best}" for r, row in df_region_summary.set_index('region_label').iterrows()}
for lib in sections_rep:
    fig = sch.plotly_spatial_visium(names, adata_vis, key='region_label', library_id=lib, hover_extra=hover, fontsize_title=20,
                                    fontsize_legend=12, fontsize_hover=13, marker_line_color="#222222",
                                    fname='fig_region_map', force=force, verbose=verbose)
    if lib == sch.params['library_main']:
        fig.show()

### Notes for interpretation

- **Composition vs expression.** Regions are compositional; LV vs RV and left vs right atrium are not separated by composition (same cell types). Use the genes (*HAND1*, *PITX2*, *BMP10*; section 8 and mp_heart_dvlp01) or the paper's clusters for laterality.
- **Hearts, not time points.** Each age is 1-3 hearts sectioned in different planes; temporal trends are suggestive. The robust ones agree across the hearts of neighbouring ages (points in section 6).
- **Deconvolution source.** The stereoscope proportions are the paper's; with the GPU cell2location map (tag gpu) the regions and these tables are recomputed in mp_heart_dvlp01 and compared with the stereoscope ones (**calc_df_c2l_vs_stereoscope**).